Bronze data processing

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/akhildataengineer@hotmail.com/Atlikon_migration_pipeline/1_setup/3_utilities

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-store-dp1/{data_source}/*.csv' #s3://sportsbar-store-dp/customers/*.csv
base_path

In [0]:
dbutils.fs.ls('s3://sportsbar-store-dp1/')

In [0]:
df = spark.read\
        .format('csv')\
        .option('header', True)\
        .option('inferSchema', True)\
        .load(base_path)\
        .withColumn("read_timestamp", F.current_timestamp())\
        .select("*", "_metadata.file_name", "_metadata.file_size")

display(df.limit(10))

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

Silver Data Processing